# Worked loan traces

One Stage 1, Stage 2 and Stage 3 loan is traced from monthly terms to final provision.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

In [2]:
summary = query('select * from worked_trace_summary order by stage'); summary

,loan_id,stage,loan_ecl,gross_exposure,current_pd_12m,origination_pd_12m,current_lifetime_pd,origination_lifetime_pd,current_ltv,current_dpd,remaining_months_to_legal_maturity,property_state,origination_date,primary_stage_reason,coverage_ratio,trace_explanation
0,RM000480,1,27.8829,"318,139.6100",0.0020,0.0010,0.0376,0.0193,57.2194,0.0000,228.0000,WA,2015-03-01,No SICR trigger,0.0001,Sum monthly MPD x LGD x EAD x DF through month...
1,RM001506,2,"12,953.9529","230,184.1000",0.3148,0.0016,1.0000,0.0514,39.0000,0.0000,390.0000,CA,2015-07-01,Prior default history,0.0563,Sum monthly MPD x LGD x EAD x DF through remai...
2,RM002463,3,"12,983.6638","205,696.7900",0.5000,0.0010,1.0000,0.0197,48.0000,210.0000,234.0000,OK,2015-09-01,Current default/credit-impaired,0.0631,Gross exposure less scenario-specific discount...


In [3]:
monthly = query('select * from worked_trace_monthly')
monthly.groupby(['loan_id','stage','scenario','scenario_weight'], as_index=False).period_ecl.sum()

,loan_id,stage,scenario,scenario_weight,period_ecl
0,RM000480,1,Base,0.5798,27.3074
1,RM000480,1,Downside,0.1693,32.9080
2,RM000480,1,Upside,0.2509,25.8209
3,RM001506,2,Base,0.5798,"12,897.8565"
4,RM001506,2,Downside,0.1693,"13,346.8613"
5,RM001506,2,Upside,0.2509,"12,818.3696"
6,RM002463,3,Base,0.5798,"12,983.6638"
7,RM002463,3,Downside,0.1693,"12,983.6638"
8,RM002463,3,Upside,0.2509,"12,983.6638"


In [4]:
reconciliation = (monthly.groupby(['loan_id','stage','scenario','scenario_weight'], as_index=False).period_ecl.sum()
.assign(weighted_ecl=lambda x: x.period_ecl*x.scenario_weight)
.groupby(['loan_id','stage'], as_index=False).weighted_ecl.sum()
.merge(summary[['loan_id','loan_ecl']], on='loan_id'))
reconciliation['difference'] = reconciliation.weighted_ecl-reconciliation.loan_ecl
reconciliation

,loan_id,stage,weighted_ecl,loan_ecl,difference
0,RM000480,1,27.8829,27.8829,0.0000
1,RM001506,2,"12,953.9529","12,953.9529",0.0000
2,RM002463,3,"12,983.6638","12,983.6638",0.0000


In [5]:
stage1_id = summary.loc[summary.stage.eq(1),'loan_id'].iloc[0]
monthly[(monthly.loan_id.eq(stage1_id)) & (monthly.scenario.eq('Base'))][
['future_month','marginal_pd','lgd','ead','discount_factor','period_ecl']].head(12)

,future_month,marginal_pd,lgd,ead,discount_factor,period_ecl
0,1,0.0002,0.0462,"317,819.6985",0.9966,2.5808
1,2,0.0002,0.0458,"316,893.2469",0.9932,2.5219
2,3,0.0002,0.0455,"315,963.6106",0.9898,2.4641
3,4,0.0002,0.0451,"315,030.7786",0.9864,2.4072
4,5,0.0002,0.0448,"314,094.7401",0.9830,2.3514
5,6,0.0002,0.0444,"313,155.4839",0.9796,2.2966
6,7,0.0002,0.0440,"312,212.9990",0.9763,2.2429
7,8,0.0002,0.0436,"311,267.2744",0.9729,2.1901
8,9,0.0002,0.0433,"310,318.2988",0.9696,2.1383
9,10,0.0002,0.0429,"309,366.0611",0.9663,2.0875


In [6]:
stage2_id = summary.loc[summary.stage.eq(2),'loan_id'].iloc[0]
monthly[(monthly.loan_id.eq(stage2_id)) & (monthly.scenario.eq('Base'))][
['future_month','conditional_pd','survival_probability','marginal_pd','lgd','ead','discount_factor','period_ecl']].head(18)

,future_month,conditional_pd,survival_probability,marginal_pd,lgd,ead,discount_factor,period_ecl
36,1,0.0314,1.0000,0.0314,0.0825,"230,345.6602",0.9965,594.3672
37,2,0.0314,0.9571,0.0300,0.0820,"230,069.6917",0.9930,562.6223
38,3,0.0314,0.9160,0.0287,0.0815,"229,792.7458",0.9894,532.5115
39,4,0.0314,0.8767,0.0275,0.0809,"229,514.8191",0.9860,503.9559
40,5,0.0314,0.8391,0.0263,0.0804,"229,235.9081",0.9825,476.8798
41,6,0.0314,0.8031,0.0252,0.0799,"228,956.0092",0.9790,451.2110
42,7,0.0314,0.7687,0.0241,0.0793,"228,675.1191",0.9756,426.8802
43,8,0.0314,0.7357,0.0231,0.0788,"228,393.2341",0.9721,403.8213
44,9,0.0319,0.7041,0.0225,0.0782,"228,110.3508",0.9687,388.7914
45,10,0.0357,0.6735,0.0240,0.0777,"227,826.4656",0.9653,410.5795


In [7]:
stage3_id = summary.loc[summary.stage.eq(3),'loan_id'].iloc[0]
monthly[monthly.loan_id.eq(stage3_id)][
['scenario','ead','projected_property_value','gross_collateral_proceeds','recovery_expenses',
 'expected_mi_recovery','expected_recovery','discount_factor','discounted_expected_recovery','period_ecl']]

,scenario,ead,projected_property_value,gross_collateral_proceeds,recovery_expenses,expected_mi_recovery,expected_recovery,discount_factor,discounted_expected_recovery,period_ecl
1206,Base,"205,696.7900","447,986.0695","394,227.7412","44,798.6070",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"
1207,Upside,"205,696.7900","461,229.6822","405,882.1204","46,122.9682",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"
1208,Downside,"205,696.7900","396,435.7715","348,863.4789","39,643.5772",0.0000,"205,696.7900",0.9369,"192,713.1262","12,983.6638"


Stage 3 has no performing-loan marginal PD. Its provision is exposure less discounted scenario-specific expected recovery.